# Parquet - Value Errors

Introduce out-of-range and/or incorrectly scaled numeric values while retaining the intended column type.

**Instructions:** Run `prepare_data_parquet.ipynb` to create clean `data/weather.parquet` first (`uv add pyarrow`). This notebook overwrites the Parquet file; restore the clean artifact before rerunning. For combination, use only the required config/function/call, and load/save once.

In [ ]:
from pathlib import Path
from random import Random

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

# Configuration
DATA_PATH = Path("data/weather.parquet")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local dates.
SEED = 814205

# Value Errors
ERROR_TARGET_COLUMN = "temperature_2m"
ERROR_KIND = "mixed"  # out_of_bounds, scaled, mixed, random
ERROR_PLACEMENT = "mixed"  # scattered, grouped, mixed, random
ERROR_RATE = (0.01, 0.03)  # Share of current eligible nonmissing/valid values changed.
ERROR_GROUPS = 3
ERROR_GROUPED_SHARE = 0.70
VALID_MIN = -60.0
VALID_MAX = 60.0
OUTSIDE_MARGIN = 10.0
SCALE_FACTOR = 2.0
ERROR_BOUNDS_SHARE = 0.50

In [ ]:
# Shared selection: include once in an assembled dirty_data.py.
def scoped_rows(df, *, time_column, timezone, start_date, end_date):
    if time_column not in df:
        raise ValueError("Time column not found.")
    times = pd.to_datetime(df[time_column], utc=True, errors="raise")
    if times.isna().any():
        raise ValueError("Time column contains missing timestamps.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    scoped = df.loc[times.ge(start) & times.lt(end)]
    if scoped.empty:
        raise ValueError("No records in the selected interval.")
    # Indices refer to the input DataFrame. No complete-hourly assumption.
    return scoped.sort_values(time_column, kind="stable").index.tolist(), start, end

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_value_errors(
    df, rng, *, time_column, timezone, start_date, end_date,
    target_column, error_kind, placement, error_rate, n_groups,
    grouped_share, valid_min, valid_max, outside_margin,
    scale_factor, bounds_share,
):
    if target_column not in df or target_column == time_column:
        raise ValueError("Select an existing non-time target column.")
    if not pd.api.types.is_numeric_dtype(df[target_column]):
        raise ValueError("Numeric value errors require a numeric target column.")
    if not valid_min < valid_max or outside_margin <= 0 or scale_factor in (0, 1):
        raise ValueError("Invalid numeric error configuration.")
    if error_kind not in ("out_of_bounds", "scaled", "mixed", "random"):
        raise ValueError("Unknown value-error kind.")
    if not 0 < bounds_share < 1:
        raise ValueError("Bounds share must be between 0 and 1.")
    dirty = df.reset_index(drop=True).copy()
    eligible, _, _ = scoped_rows(
        dirty, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    # Preserve nulls already generated by a preceding missing-values function.
    eligible = [i for i in eligible if pd.notna(dirty.at[i, target_column])
                and valid_min <= dirty.at[i, target_column] <= valid_max]
    kind = (rng.choice(("out_of_bounds", "scaled", "mixed"))
            if error_kind == "random" else error_kind)
    if kind in ("scaled", "mixed"):
        eligible = [i for i in eligible if dirty.at[i, target_column] != 0]
    if pd.api.types.is_integer_dtype(dirty[target_column]):
        # No accidental integer->float schema change: separate schema testing owns that.
        if not all(float(v).is_integer() for v in (valid_min, valid_max, outside_margin, scale_factor)):
            raise ValueError("Integer field needs integral bounds, margin and scale factor.")
    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=error_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    if kind == "mixed" and len(selected) < 2:
        raise ValueError("Mixed errors require at least two changed values.")
    count = len(selected)
    bounds_count = (count if kind == "out_of_bounds" else 0 if kind == "scaled" else
                    max(1, min(count - 1, round(count * bounds_share))))
    outside = set(rng.sample(selected, k=bounds_count))
    for pos in selected:
        i = eligible[pos]
        old = dirty.at[i, target_column]
        if pos in outside:
            new = (valid_min - outside_margin if rng.random() < 0.5
                   else valid_max + outside_margin)
        else:
            new = old * scale_factor
        if new == old:
            raise ValueError("Configured error did not change a selected value.")
        dirty.at[i, target_column] = new
    return dirty

In [ ]:
# Small controlled trial (no source file changed).
trial_times = pd.date_range("2025-08-01", periods=7*24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({
    "time": trial_times,
    "temperature_2m": [float(10 + i % 17) for i in range(len(trial_times))],
    "relative_humidity_2m": pd.Series([int(40 + i % 40) for i in range(len(trial_times))], dtype="int64"),
    "precipitation": [float(i % 10) for i in range(len(trial_times))],
})

trial_dirty = add_value_errors(trial, Random(SEED),
    time_column="time", timezone=TIMEZONE, start_date="2025-08-01", end_date="2025-08-08",
    target_column="temperature_2m", error_kind="mixed", placement="mixed",
    error_rate=(0.10, 0.10), n_groups=2, grouped_share=0.70,
    valid_min=-60, valid_max=60, outside_margin=10, scale_factor=2,
    bounds_share=0.50)
assert (trial_dirty["temperature_2m"] != trial["temperature_2m"]).sum() == round(len(trial)*0.10)
assert trial_dirty["temperature_2m"].dtype == trial["temperature_2m"].dtype
trial_dirty.loc[trial_dirty["temperature_2m"] != trial["temperature_2m"]].head()

In [ ]:
# Verify actual Parquet types and values on a small temporary file.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    clean_path = Path(temporary) / "clean.parquet"
    temp_path = Path(temporary) / "trial.parquet"
    trial.to_parquet(clean_path, index=False)
    trial_dirty.to_parquet(temp_path, index=False)
    roundtrip = pd.read_parquet(temp_path)
    before_schema = pq.read_schema(clean_path)
    after_schema = pq.read_schema(temp_path)
    assert len(roundtrip) == len(trial_dirty)
    assert roundtrip.columns.tolist() == trial_dirty.columns.tolist()
    assert pq.read_metadata(temp_path).num_rows == len(trial_dirty)
    assert all(before_schema.field(c).type == after_schema.field(c).type for c in trial.columns)
    print("Stored Parquet schema:", after_schema)

In [ ]:
# Apply the reusable function to a freshly prepared Parquet artifact.
clean = pd.read_parquet(DATA_PATH)
rng = Random(SEED)
dirty = add_value_errors(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    target_column=ERROR_TARGET_COLUMN, error_kind=ERROR_KIND,
    placement=ERROR_PLACEMENT, error_rate=ERROR_RATE,
    n_groups=ERROR_GROUPS, grouped_share=ERROR_GROUPED_SHARE,
    valid_min=VALID_MIN, valid_max=VALID_MAX,
    outside_margin=OUTSIDE_MARGIN, scale_factor=SCALE_FACTOR,
    bounds_share=ERROR_BOUNDS_SHARE,
)
dirty.to_parquet(DATA_PATH, index=False)
saved = pd.read_parquet(DATA_PATH)

In [ ]:
# Inspect: out-of-range values, then compare clean and dirty measurements.
invalid = saved[ERROR_TARGET_COLUMN].notna() & ~saved[ERROR_TARGET_COLUMN].between(VALID_MIN, VALID_MAX)
saved.loc[invalid, [TIME_COLUMN, ERROR_TARGET_COLUMN]].head(15)

In [ ]:
# Verify persistence: all changes exist after writing the Parquet file.
assert len(saved) == len(dirty)
assert saved.columns.tolist() == dirty.columns.tolist()
assert pq.read_metadata(DATA_PATH).num_rows == len(saved)
changed = saved[ERROR_TARGET_COLUMN].ne(clean[ERROR_TARGET_COLUMN])
assert changed.any() and len(saved) == len(clean)
pd.testing.assert_frame_equal(saved.drop(columns=ERROR_TARGET_COLUMN),
                              clean.drop(columns=ERROR_TARGET_COLUMN), check_dtype=False)
print("Changed numeric values:", int(changed.sum()))

In [ ]:
# Changed values against the clean reference; show one configured month.
changed = saved[ERROR_TARGET_COLUMN].ne(clean[ERROR_TARGET_COLUMN])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(clean[TIME_COLUMN], clean[ERROR_TARGET_COLUMN], linewidth=0.7, label="Clean")
ax.scatter(saved.loc[changed, TIME_COLUMN], saved.loc[changed, ERROR_TARGET_COLUMN],
           s=12, label="Changed", zorder=3)
ax.set(title=f"Injected value errors: {ERROR_TARGET_COLUMN}", xlabel="Time", ylabel=ERROR_TARGET_COLUMN)
ax.legend(); plt.tight_layout(); plt.show()